In [ ]:
from google.colab import files
files.upload()

In [ ]:
!pip install -q kaggle

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
!kaggle datasets download -d bjoernjostein/physionet-challenge-2016 -p /content
!unzip -q /content/physionet-challenge-2016.zip -d /content/physionet2016


In [ ]:
!pip install -q tensorflow librosa scipy scikit-learn pandas


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from scipy.signal import find_peaks
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils import class_weight


In [ ]:
SR = 4000
DURATION = 5
N_MFCC = 40
MAX_LEN = 216


In [ ]:
def extract_mfcc(file):
    audio, _ = librosa.load(file, sr=SR, duration=DURATION)

    mfcc = librosa.feature.mfcc(y=audio, sr=SR, n_mfcc=N_MFCC)
    mfcc = mfcc.T

    if mfcc.shape[0] < MAX_LEN:
        mfcc = np.pad(mfcc, ((0, MAX_LEN - mfcc.shape[0]), (0, 0)))
    else:
        mfcc = mfcc[:MAX_LEN]

    return mfcc


In [ ]:
X = []
y = []

base_path = "/content/physionet2016"

reference_files = glob.glob(base_path + "/**/REFERENCE.csv", recursive=True)
print("Found reference files:", reference_files)

for ref_file in reference_files:
    df = pd.read_csv(ref_file, header=None)
    folder = os.path.dirname(ref_file)

    for _, row in df.iterrows():
        file_id = row[0]
        label = row[1]
        wav_path = os.path.join(folder, file_id + ".wav")

        if not os.path.exists(wav_path):
            continue

        X.append(extract_mfcc(wav_path))

        if label == -1:
            y.append(0)  # Normal
        else:
            y.append(1)  # Abnormal

X = np.array(X)[..., np.newaxis]
y = np.array(y)

print("Total samples:", X.shape[0])
print("Input shape:", X.shape)


Found reference files: ['/content/physionet2016/training-b/REFERENCE.csv', '/content/physionet2016/training-c/REFERENCE.csv', '/content/physionet2016/validation/REFERENCE.csv', '/content/physionet2016/training-f/REFERENCE.csv', '/content/physionet2016/training-d/REFERENCE.csv', '/content/physionet2016/training-a/REFERENCE.csv', '/content/physionet2016/training-e/REFERENCE.csv']
Total samples: 3541
Input shape: (3541, 216, 40, 1)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = {0: weights[0], 1: weights[1]}
print("Class weights:", class_weights)


Class weights: {0: np.float64(0.6498393758604865), 1: np.float64(2.1684532924961717)}


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=X.shape[1:]),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(2, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 214, 38, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 107, 19, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 105, 17, 64)    │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 52, 8, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 26624)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,408,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,427,074 (13.07 MB)

 Trainable params: 3,427,074 (13.07 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=30,
    batch_size=32,
    class_weight=class_weights
)


Epoch 1/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 16s 107ms/step - accuracy: 0.6488 - loss: 1.6948 - val_accuracy: 0.8082 - val_loss: 0.3822
Epoch 2/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - accuracy: 0.7976 - loss: 0.3696 - val_accuracy: 0.8533 - val_loss: 0.3228
Epoch 3/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.8492 - loss: 0.3186 - val_accuracy: 0.8265 - val_loss: 0.3765
Epoch 4/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.8526 - loss: 0.2766 - val_accuracy: 0.8378 - val_loss: 0.3198
Epoch 5/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8556 - loss: 0.2843 - val_accuracy: 0.8420 - val_loss: 0.3537
Epoch 6/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8563 - loss: 0.2694 - val_accuracy: 0.8688 - val_loss: 0.3010
Epoch 7/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.8811 - loss: 0.2370 - val_accuracy: 0.8604 - val_loss: 0.3417
Epoch 8/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.8709 - loss: 0.2322 - val_accuracy: 0.8378 -

In [ ]:
y_pred = np.argmax(model.predict(X_test), axis=1)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Normal", "Abnormal"]))


23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step
Confusion Matrix:
[[496  50]
 [ 30 133]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.94      0.91      0.93       546
    Abnormal       0.73      0.82      0.77       163

    accuracy                           0.89       709
   macro avg       0.83      0.86      0.85       709
weighted avg       0.89      0.89      0.89       709



In [ ]:
def detect_arrhythmia(audio):
    peaks, _ = find_peaks(audio, distance=SR*0.3)
    if len(peaks) < 3:
        return False
    intervals = np.diff(peaks)
    variability = np.std(intervals) / np.mean(intervals)
    return variability > 0.25


In [ ]:
def detect_murmur(audio):
    S = np.abs(librosa.stft(audio))
    freqs = librosa.fft_frequencies(sr=SR)
    murmur_band = (freqs > 150) & (freqs < 600)
    energy_ratio = S[murmur_band].sum() / S.sum()
    return energy_ratio > 0.35


In [ ]:
def classify_heart_sound(file):
    audio, _ = librosa.load(file, sr=SR, duration=DURATION)
    mfcc = extract_mfcc(file)[np.newaxis, ..., np.newaxis]
    pred = np.argmax(model.predict(mfcc))

    if pred == 0:
        return "Normal"
    if detect_arrhythmia(audio):
        return "Arrhythmia"
    if detect_murmur(audio):
        return "Murmur"
    return "Abnormal"


In [ ]:
sample = glob.glob("/content/physionet2016/**/c0009.wav", recursive=True)[0]
print(classify_heart_sound(sample))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
Abnormal


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

with open("heart_model1.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved")


Saved artifact at '/tmp/tmpw9iog13j'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 216, 40, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  138633157950352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138633157939216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138633157945744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138633157947856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138633157948624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138633157942672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138633157939792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138633157944592: TensorSpec(shape=(), dtype=tf.resource, name=None)
TFLite model saved


In [ ]:
from google.colab import files
files.download("heart_model1.tflite")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>